In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.add_import("@ml_db.ml_schema.ml_stage/empulse-0.10.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl")



In [ ]:
session.file.get("@ml_db.ml_schema.ml_stage/empulse-0.10.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "/tmp/")
!pip install --no-deps /tmp/empulse-0.10.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
import empulse

In [ ]:
import sys
from unittest.mock import MagicMock
for mod in ['imblearn', 'imblearn.base', 'imblearn.pipeline', 'imblearn.over_sampling', 'imblearn.under_sampling', 'imblearn.utils']:
    sys.modules[mod] = MagicMock()

from empulse.models import ProfLogitClassifier
from empulse.metrics import empc_score
from empulse.optimizers import Generation
from scipy.optimize import OptimizeResult

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.table('ML_DB.ML_SCHEMA.CHURN').to_pandas()
train_df = session.table('ML_DB.ML_SCHEMA.TRAIN').to_pandas()
test_df = session.table('ML_DB.ML_SCHEMA.TEST').to_pandas()

In [ ]:
#define X Y

x_train = train_df.drop(['STATUS', 'SIX_MONTHS_TOTAL_REVENUE_ORIG', 'TENURE_MONTHS_ORIG', 'CLV'],axis=1)
y_train = train_df['STATUS']
clv_train = train_df['CLV']

In [ ]:
x_test = test_df.drop(['STATUS', 'SIX_MONTHS_TOTAL_REVENUE_ORIG', 'TENURE_MONTHS_ORIG', 'CLV'],axis=1)
y_test = test_df['STATUS']
clv_test = test_df['CLV']

### fitting logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
modellr=LogisticRegression(solver='liblinear')

modellr.fit(x_train,y_train)

y_predlr = modellr.predict(x_test)
y_problr = modellr.predict_proba(x_test)[:, 1]

In [ ]:
y_predtrainlr = modellr.predict(x_train)
y_probtrainlr = modellr.predict_proba(x_train)[:, 1]

### Fitting proflogit with CLV

In [ ]:
model = ProfLogitClassifier(
    loss=empc_score,
    C=1,
    l1_ratio=1,
    optimizer_params={
        "max_iter": 10000,       
        "patience": 50,        
        "tolerance": 1e-3,     
        "bounds": (-3, 3)      
    },
    n_jobs=3              
)

In [ ]:
model.fit(x_train, y_train,clv = clv_train)

In [ ]:
y_predpl = model.predict(x_test)
y_probpl = model.predict_proba(x_test)[:, 1]

In [ ]:
y_predtrainpl = model.predict(x_train)
y_probtrainpl = model.predict_proba(x_train)[:, 1]

In [ ]:
clv_score = 2280.744

# Evaluate PROFLOGIT model

In [ ]:
from empulse.metrics import empc
params_pl_train = {
    'clv': clv_score,
    'incentive_cost': 1000,
    'contact_cost': 10
}
print(empc(y_train, y_probtrainpl, **params_pl_train))

In [ ]:
from empulse.metrics import empc
params_pl_test = {
    'clv': clv_score,
    'incentive_cost': 1000,
    'contact_cost': 10
}

print(empc(y_test, y_probpl, **params_pl_test))

In [ ]:
from sklearn.metrics import f1_score
print(f1_score(y_test, y_predpl, average=None))

In [ ]:
from sklearn.metrics import accuracy_score
accuracypl = accuracy_score(y_test, y_predpl)
print("Model Accuracy:", accuracypl)

# Evaluate Logisticregression model

In [ ]:
from empulse.metrics import empc
params_lr_train = {
    'clv': clv_score,
    'incentive_cost': 1000,
    'contact_cost': 10
}
print(empc(y_train, y_probtrainlr, **params_lr_train))

In [ ]:
params_lr_test = {
    'clv': clv_score,
    'incentive_cost': 1000,
    'contact_cost': 10
}
print(empc(y_test, y_problr, **params_lr_test))

In [ ]:
print(f1_score(y_test, y_predlr, average=None))

In [ ]:
accuracylr = accuracy_score(y_test, y_predlr)
print("Model Accuracy:", accuracylr)

## 5 x 2 Cross Validation (proflogit)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Lambda values
lambda_grid = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]

# Store results
all_results = []

# Fixed parameters
clv_val = 2280.744
d_val = 1000
f_val = 10

for lam in lambda_grid:
    
    print(f"\n========== Lambda = {lam} ==========")
    
    # Convert lambda to C
    C_val = 1 / lam
    
    for rep in range(1, 6):
        
        print(f"  Repetition {rep} ... ", end="")
        
        # Random 50/50 split
        X_a, X_b, y_a, y_b = train_test_split(
            x_train,
            y_train,
            test_size=0.5,
            random_state=None,
            stratify=y_train
        )
        
        
        # Round A: Train on fold_a, test on fold_b
        
        
        model_a = ProfLogitClassifier(
            loss=empc_score,
            C=C_val,
            l1_ratio=1,
            optimizer_params={
                "max_iter": 10000,
                "patience": 50,
                "tolerance": 1e-3,
                "bounds": (-3, 3)
            },
            n_jobs=3
        )
        
        model_a.fit(X_a, y_a)
        
        pred_a = model_a.predict_proba(X_b)[:, 1]
        
        empc_a = empc_score(
            y_b,
            pred_a,
            clv=clv_val,
            incentive_cost=d_val,
            contact_cost=f_val
        )
        
        all_results.append({
            "lambda": lam,
            "repetition": rep,
            "fold": "A",
            "empc": empc_a
        })
        
        
        # Round B: Train on fold_b, test on fold_a
       
        
        model_b = ProfLogitClassifier(
            loss=empc_score,
            C=C_val,
            l1_ratio=1,
            optimizer_params={
                "max_iter": 10000,
                "patience": 50,
                "tolerance": 1e-3,
                "bounds": (-3, 3)
            },
            n_jobs=3
        )
        
        model_b.fit(X_b, y_b)
        
        pred_b = model_b.predict_proba(X_a)[:, 1]
        
        empc_b = empc_score(
            y_a,
            pred_b,
            clv=clv_val,
            incentive_cost=d_val,
            contact_cost=f_val
        )
        
        all_results.append({
            "lambda": lam,
            "repetition": rep,
            "fold": "B",
            "empc": empc_b
        })
        
        print(
            f"Fold A EMPC = {empc_a:.2f} | "
            f"Fold B EMPC = {empc_b:.2f}"
        )

# Convert to dataframe
all_results = pd.DataFrame(all_results)

In [ ]:
all_results

In [ ]:
summary_table = (
    all_results
    .groupby("lambda")["empc"]
    .agg(
        mean="mean",
        sd="std",
        median="median",
        min="min",
        max="max"
    )
    .reset_index()
)

# rounding like 2 decimals
summary_table[["mean", "sd", "median", "min", "max"]] = \
    summary_table[["mean", "sd", "median", "min", "max"]].round(2)

summary_table = summary_table.rename(columns={"lambda": "Lambda"})

print("\n========== SUMMARY ==========")
print(summary_table)

In [ ]:
best_row = summary_table.loc[summary_table["mean"].idxmax()]
best_lambda = best_row["Lambda"]

print("\nOptimal Lambda:", best_lambda)

In [ ]:
all_results["lambda_label"] = all_results["lambda"].apply(lambda x: f"λ = {x:g}")

label_order = ["λ = " + str(l) for l in lambda_grid]
all_results["lambda_label"] = pd.Categorical(
    all_results["lambda_label"],
    categories=label_order,
    ordered=True
)

In [ ]:
means = (
    all_results
    .groupby("lambda_label")["empc"]
    .mean()
    .reindex(label_order)
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

# Prepare boxplot data in order
data_to_plot = [
    all_results.loc[all_results["lambda_label"] == lbl, "empc"].values
    for lbl in label_order
]

# Boxplot
plt.boxplot(
    data_to_plot,
    labels=label_order,
    patch_artist=True,
    showfliers=True
)

# Mean line + points (red)
plt.plot(
    range(1, len(label_order) + 1),
    means.values,
    color="red",
    marker="o",
    linewidth=1.5
)

plt.scatter(
    range(1, len(label_order) + 1),
    means.values,
    color="red"
)

plt.xlabel("")
plt.ylabel("EMPC (LKR)")
plt.title("Average EMPC for various values of ProfTree's λ")

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()